# Mean-field applications: TDA, RPA and QRPA

Companion notebook to Chapter 7 of *Quantum mechanics for many-particle
systems*.  Every number quoted in the chapter is produced here; the code is
the same as in `BookManybody/BookMaterial/Programs/rpa.py`.

The model has $L$ doubly degenerate, equally spaced levels $p=1,\dots,L$ with
spin $\sigma=\pm$, holding $N$ fermions:

$$\hat H_0 = \xi\sum_{p\sigma}(p-1)\,a^\dagger_{p\sigma}a_{p\sigma},$$
$$\hat V_{\rm pair} = -\frac{g}{2}\sum_{pq}
   a^\dagger_{p+}a^\dagger_{p-}a_{q-}a_{q+},\qquad
  \hat V_{\rm ph} = -\frac{f}{2}\sum_{pqr}
   \left(a^\dagger_{p+}a^\dagger_{p-}a_{q-}a_{r+} + {\rm h.c.}\right).$$

At $f=0$ this is exactly the pairing model of Chapter 4.  The particle-hole
term breaks pairs, which is what the pairing term cannot do.

Contents:

1. The model and its exact spectrum
2. The Hartree-Fock reference
3. Validating the $A$ and $B$ matrices
4. Tamm-Dancoff and RPA
5. RPA and the stability of the mean field
6. BCS
7. Quasiparticle RPA, the spurious mode and the mode content
8. Everything against everything

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join("..", "BookManybody", "BookMaterial", "Programs"))
import rpa

np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. The model and its exact spectrum

Exact diagonalisation is Chapter 5 applied without modification: determinants
are bit strings, the Hamiltonian is built by applying each term to every
basis state, and the lowest eigenvalues follow from Chapter 1's algorithms.
Restricting to the balanced $S_z=0$ sector cuts the dimension from
$\binom{8}{4}=70$ to 36.

In [ ]:
rpa.demo_model()

In [ ]:
gs = np.linspace(-1.0, 1.0, 41)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for alpha, panel in ((0.05, 0), (0.5, 1)):
    data = np.array([rpa.PairingPH(levels=4, particles=4,
                                   g=g, f=alpha * g).spectrum(6) for g in gs])
    for m in range(data.shape[1]):
        ax[panel].plot(gs, data[:, m], lw=1.4)
    ax[panel].set_xlabel("$g$"); ax[panel].set_ylabel("$E_k$")
    ax[panel].set_title(rf"exact eigenvalues, $f = {alpha}\,g$")
    ax[panel].grid(alpha=0.3)
fig.tight_layout(); plt.show()

The weak-$f$ panel shows the near-degenerate pairs of the pairing model,
split only slightly.  At $f = 0.5\,g$ the pair-breaking term reorganises the
low-lying structure and levels cross.

## 2. The Hartree-Fock reference

Both approximations are built on a stationary mean field.  With pure pairing
the filled Fermi sea is already the solution, as Chapter 6 found; the
particle-hole term makes the iteration non-trivial.

In [ ]:
rpa.demo_hartree_fock()

## 3. Validating the $A$ and $B$ matrices

Everything is computed as literal double commutators

$$A_{KL} = \langle 0|[\hat O_K^\dagger,[\hat H,\hat O_L]]|0\rangle,\qquad
  B_{KL} = -\langle 0|[\hat O_K^\dagger,[\hat H,\hat O_L^\dagger]]|0\rangle,$$

so it is worth checking against the textbook expressions.  The important
identity is the first one below: the Tamm-Dancoff matrix **is** the
Hamiltonian restricted to the $1p$-$1h$ block, measured from the reference.
TDA is CIS.

In [ ]:
rpa.demo_validation()

## 4. Tamm-Dancoff and RPA

$$\hat Q^\dagger_\nu = \sum_{mi}\left(X^\nu_{mi}a^\dagger_m a_i
  - Y^\nu_{mi}a^\dagger_i a_m\right),\qquad
\begin{pmatrix}A & B\\ -B^* & -A^*\end{pmatrix}
\begin{pmatrix}X^\nu\\ Y^\nu\end{pmatrix}
= \omega_\nu\begin{pmatrix}X^\nu\\ Y^\nu\end{pmatrix}.$$

Setting $B=0$ recovers TDA.  The RPA correlation energy is

$$E^{\rm RPA}_{\rm corr} = \tfrac12\left(\sum_\nu\omega_\nu - {\rm Tr}\,A\right).$$

In [ ]:
rpa.demo_tda_rpa()

In [ ]:
fock = rpa.FockSpace(4)
gs = np.linspace(0.2, 2.0, 13)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
for frac, style in ((0.05, "-"), (0.5, "--")):
    e0, e_rpa, w_ex, w_tda, w_rpa = [], [], [], [], []
    for g in gs:
        f = frac * g
        ex = rpa.PairingPH(levels=4, particles=4, g=g, f=f).spectrum(2)
        r = rpa.tda_rpa(fock, 4, g, f)
        e0.append(ex[0]); e_rpa.append(r["hf"] + r["ecorr"])
        w_ex.append(ex[1] - ex[0]); w_tda.append(r["tda"][0])
        w_rpa.append(r["rpa"][0])
    lab = f"$f = {frac}g$"
    ax[0].plot(gs, e0, "k" + style, label=f"exact, {lab}")
    ax[0].plot(gs, e_rpa, "C0" + style, label=f"HF+RPA, {lab}")
    ax[1].plot(gs, w_ex, "k" + style, label=f"exact, {lab}")
    ax[1].plot(gs, w_rpa, "C0" + style, label=f"RPA, {lab}")
    ax[1].plot(gs, w_tda, "C3" + style, alpha=0.7, label=f"TDA, {lab}")
ax[0].set_xlabel("$g$"); ax[0].set_ylabel("ground-state energy")
ax[0].set_title("ground state"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("$g$"); ax[1].set_ylabel(r"$\omega_1$")
ax[1].set_title("lowest excitation"); ax[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

RPA lowers the energy below Hartree-Fock and overshoots the exact ground
state -- the quasiboson approximation over-counts the correlations.  Both TDA
and RPA put the lowest excitation well below the exact one, and the gap widens
with the pairing strength: the particle-hole phonon knows nothing about
pairing correlations.

### A closed form

For pure pairing the whole particle-hole problem collapses and

$$\omega^{\rm TDA} = \xi + \frac{g}{2},\qquad
  \omega^{\rm RPA} = \sqrt{\xi^2 + g\xi}
   = \sqrt{\left(\xi+\tfrac{g}{2}\right)^2 - \left(\tfrac{g}{2}\right)^2},$$

the classic $\sqrt{A^2-B^2}$ of a two-level RPA problem.  Note that the root
is real for every $g > -\xi$: the particle-hole channel of the pairing model
never goes unstable.

In [ ]:
gs = np.linspace(0.0, 6.0, 25)
solved = [rpa.tda_rpa(fock, 4, g, 0.0) for g in gs]
tda = [s["tda"][0] for s in solved]
rr  = [s["rpa"][0] for s in solved]

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(gs, tda, "C3o", ms=3, label="TDA (computed)")
ax.plot(gs, 1.0 + 0.5 * gs, "C3-", lw=1, label=r"$\xi + g/2$")
ax.plot(gs, rr, "C0s", ms=3, label="RPA (computed)")
ax.plot(gs, np.sqrt(1.0 + gs), "C0-", lw=1, label=r"$\sqrt{\xi^2+g\xi}$")
ax.set_xlabel("$g$"); ax.set_ylabel(r"$\omega_1$")
ax.legend(); fig.tight_layout(); plt.show()

## 5. RPA and the stability of the mean field

Chapter 6's stability matrix and the RPA matrix are built from the *same*
blocks, differently arranged:

$$\hat M_{\rm stab} = \begin{pmatrix}A & B\\ B^* & A^*\end{pmatrix}
  \ ({\rm Hermitian}),\qquad
  \hat M_{\rm RPA} = \begin{pmatrix}A & B\\ -B^* & -A^*\end{pmatrix}
  = \eta\,\hat M_{\rm stab}.$$

The RPA frequencies are real exactly when $\hat M_{\rm stab}\ge 0$, that is,
exactly when the mean field is a local minimum.  The Lipkin model shows the
collective root going soft at $\chi=1$ and turning imaginary beyond.

In [ ]:
rpa.demo_stability()

In [ ]:
chis = np.linspace(0.02, 1.8, 60)
lowest, omega = [], []
for chi in chis:
    A, B = rpa._lipkin_blocks(N=4, eps=1.0, V=chi / 3.0)
    stab = np.block([[A, B], [B.conj(), A.conj()]])
    lowest.append(np.linalg.eigvalsh(0.5 * (stab + stab.T))[0])
    roots, n_imag = rpa.solve_rpa(A, B)
    omega.append(roots[0] if (len(roots) and n_imag == 0) else np.nan)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(chis, lowest, label=r"$\lambda_{\min}(\hat M_{\rm stab})$")
ax.plot(chis, omega, label=r"lowest RPA root $\omega_1$")
ax.plot(chis, np.sqrt(np.clip(1 - chis**2, 0, None)), "k--", lw=0.9,
        label=r"$\varepsilon\sqrt{1-\chi^2}$")
ax.axhline(0, color="k", lw=0.8); ax.axvline(1.0, color="k", lw=0.8, ls=":")
ax.set_xlabel(r"$\chi$"); ax.set_title("Lipkin: stability and the soft mode")
ax.legend(); fig.tight_layout(); plt.show()

## 6. BCS

The pairing mean field is not of Hartree-Fock type: the object that acquires
an expectation value is $\langle P^\dagger_p\rangle$, which does not conserve
particle number.

$$|{\rm BCS}\rangle = \prod_p\left(u_p + v_p\,
  a^\dagger_{p+}a^\dagger_{p-}\right)|0\rangle,\qquad u_p^2+v_p^2=1.$$

Because the trial state is a product over levels the energy is available in
closed form, and it turns out to depend on $g$ and $f$ only through
$g + 2f$: seen from the BCS vacuum the particle-hole term *is* extra pairing.

In [ ]:
rpa.demo_bcs()

In [ ]:
gs = np.linspace(0.2, 2.4, 23)
gap, ebcs, ehf, eex = [], [], [], []
for g in gs:
    b = rpa.BCS(4, 4, g, 0.0); b.solve()
    gap.append(b.gap); ebcs.append(b.E)
    ehf.append(rpa.tda_rpa(fock, 4, g, 0.0)["hf"])
    eex.append(rpa.PairingPH(levels=4, particles=4, g=g, f=0.0).spectrum(1)[0])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(gs, gap, "C2o-", ms=3)
ax[0].axvline(1.07037, color="k", ls=":", lw=0.9)
ax[0].set_xlabel("$g$"); ax[0].set_ylabel(r"$\Delta$")
ax[0].set_title(r"pairing gap; $g_c = 1.07037$")
ax[1].plot(gs, eex, "k-", label="exact")
ax[1].plot(gs, ehf, "C3--", label="Hartree-Fock")
ax[1].plot(gs, ebcs, "C2-", label="BCS")
ax[1].axvline(1.07037, color="k", ls=":", lw=0.9)
ax[1].set_xlabel("$g$"); ax[1].set_ylabel("ground-state energy")
ax[1].set_title("the transition is a mean-field artefact")
ax[1].legend()
fig.tight_layout(); plt.show()

The BCS transition at $g_c = 1.07$ is *sharp* even though the system is
finite -- but the exact curve runs straight through it with no feature at all.
Mean-field phase transitions in finite systems are approximations to smooth
crossovers.

## 7. Quasiparticle RPA

QRPA builds phonons out of pairs of Bogoliubov quasiparticles,

$$\hat Q^\dagger_\nu = \sum_{a<b}\left(X^\nu_{ab}
  \alpha^\dagger_a\alpha^\dagger_b - Y^\nu_{ab}\alpha_b\alpha_a\right),$$

with the same matrix structure.  Two features need care: the **spurious
Goldstone mode** from the broken $U(1)$ symmetry, and the **mode content**,
since quasiparticles mix particle number.

In [ ]:
rpa.demo_qrpa()

The near-zero root has unit overlap with the amplitudes built from $\hat N$,
identifying it beyond doubt.  A small *imaginary* spurious root is a numerical
artefact of the singular RPA metric, not an instability — this is easy to
mistake, and the overlap test is the way to tell the two apart.  Once the mode
is removed the QRPA matrix has no imaginary eigenvalues anywhere in the
superfluid regime.

## 8. Everything against everything

In [ ]:
rpa.demo_comparison()

In [ ]:
gs = np.linspace(0.4, 2.0, 13)
eex, ehf, erpa, ebcs, eqrpa = [], [], [], [], []
for g in gs:
    eex.append(rpa.PairingPH(levels=4, particles=4, g=g, f=0.0).spectrum(1)[0])
    r = rpa.tda_rpa(fock, 4, g, 0.0)
    ehf.append(r["hf"]); erpa.append(r["hf"] + r["ecorr"])
    q = rpa.qrpa(fock, 4, g, 0.0)
    ebcs.append(q["energy"])
    eqrpa.append(q["energy"] + q["ecorr"] if q["stable"] else np.nan)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(gs, eex, "k-o", ms=4, label="exact")
ax.plot(gs, ehf, "C3--", label="Hartree-Fock")
ax.plot(gs, erpa, "C0-s", ms=4, label="HF + ph-RPA")
ax.plot(gs, ebcs, "C2--", label="BCS")
ax.plot(gs, eqrpa, "C2-^", ms=5, label="BCS + QRPA")
ax.set_xlabel("$g$"); ax.set_ylabel("ground-state energy")
ax.set_title("pure pairing, $N = L = 4$")
ax.legend(); fig.tight_layout(); plt.show()

Particle-hole RPA tracks the exact curve closely and crosses it near
$g\approx1.8$.  BCS improves on Hartree-Fock only above $g_c$, and only
modestly.  QRPA on top of BCS overshoots badly, because the two-quasiparticle
space contains pairing vibrations reaching towards $N\pm2$ and the standard
correlation-energy formula counts every root.  Particle-number projection is
needed before BCS + QRPA becomes quantitative for a system this small; in a
heavy nucleus, where the relative number fluctuation is small, the same
machinery is far better behaved.

## The full program

Everything above lives in `BookManybody/BookMaterial/Programs/rpa.py`, which
runs as a script and prints all eight demonstrations of the chapter.

In [ ]:
print(open(rpa.__file__).read())